<h1><center>🧠 Homework №3 — RAG

*Natural Language Processing Course, HSE 2026*

In this notebook, you will build a Retrieval-Augmented Generation (RAG) system based on the lecture materials from [Evgeny Sokolov’s Classical Machine Learning course](https://github.com/esokolov/ml-course-hse/tree/master/2025-fall/lecture-notes).
Your goal is to build a RAG system that answers machine learning questions using the lecture materials as a knowledge base.


### 📥 General Rules and Submission Guidelines

1. Copying code from external sources (**including using LLMs**) without explicit citation is strictly prohibited and will result in 0 points for the entire assignment. If you consult any resources or AI tools, you must clearly state this in a separate Markdown cell. If suspected of this, you might be asked to explain your code to the grader and answer their questions during a separate session to avoid the mentioned penalty.
2. All results must be fully **reproducible**. You are required to use `set_seed` everywhere so that the grader can obtain the same results when rerunning your notebook.
3. The notebook must run from top to bottom without errors. Submissions that fail to execute sequentially will not be accepted.
4. You must satisfy all requirements in each task to receive full credit. Partial completion may lead to partial scoring.
5. Do not modify the original notebook structure or provided Markdown cells. If you choose to do so, leave an explanation as to what and why was changed. You are only allowed to write code in the sections marked `# TODO: your code here`. Any explanations, interpretations, or additional comments must be placed **in separate Markdown cells**.
6. The final submission must be a completed `.ipynb` Jupyter Notebook. You may conduct your experiments in Jupyter Notebook, VS Code, or Google Colab — whichever environment you prefer.

### 🔧 Environment Setup

Loading necessary libraries. If you need anything else, feel free to add more libraries and dependencies.

In [ ]:
!pip install pydantic pydantic-core langchain-core
!pip install langchain langchain-community langchain-openai pypdf chromadb datasets python-dotenv -q

!pip -q install -U  pypdf sentence-transformers transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline
from transformers import GenerationConfig
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"   # for Chroma
os.environ["CHROMA_TELEMETRY"] = "False"       # just in case
from glob import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.embeddings import Embeddings
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def set_seed(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1337)

### ⚙️ Data Preparation ($1$ point)

1. Download at least five lecture PDFs from [the course repository](https://github.com/esokolov/ml-course-hse/tree/master/2025-fall/lecture-notes).
 2. If needed, merge them into one combined PDF.
 3. Split the documents into chunks using the chunking method demonstrated during the seminar.
 4. Generate embeddings for the text chunks using the FRIDA model (which works well for Russian documents) - **use the FRIDA embedding wrapper provided in the seminar notebook.**
 5. Store the chunks and their embeddings in a Chroma vector database (see examples below).

Examples from the seminar:

In [ ]:
# split the documents into chunks
#text_splitter = CharacterTextSplitter(chunk_size=256, chunk_overlap=0)
splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=64)
texts = splitter.split_documents(documents)

In [ ]:
class FridaEmbeddings(Embeddings):

    def __init__(self, model_name = "ai-forever/FRIDA"):
        # todo

        # device choosing, using RetrievalQA.from_chain_type()
        # todo

    def embed_documents(self, texts):
        # todo

    def embed_query(self, text):
        return ... # todo

In [ ]:
db = Chroma.from_documents(texts, embeddings)

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

###🚀 Baseline RAG System ($1$ point)

Implement a baseline RAG system similar to the one demonstrated during the seminar.

Use the following components:
 * Vector database: `Chroma`
 * Embedding model: `FRIDA`
 * LLM: `Qwen2.5-1.5B-Instruct`

The system should retrieve relevant lecture chunks and generate answers using the LLM.

In [ ]:
# TODO: your code here

### 🔬 **Experimental Design**

(*This is the MOST IMPORTANT research part of the assignment*)

You must design and justify an evaluation framework for your RAG system.

Your evaluation should include a dataset of questions and expected answers.


#### 📊 **a) Create an Evaluation Dataset** ($2.5$ points)

Construct a dataset containing $10$–$20$ Machine Learning questions along with short reference answers.

You may create the questions using one of the following approaches:
 * Write the questions manually
 * Collect questions from ML forums
 * Use other reliable sources

For each question, provide a short expected answer that the RAG system should produce.

Additionally, you must describe the dataset creation process in detail, just as a real ML engineer would:
 * Explain how you collected or designed the questions
 * Justify why this approach was chosen
 * Describe any filtering or selection criteria

**`⚠️ Important:`**
Copying datasets from other students is **strictly prohibited**.
If two students submit identical datasets, both submissions will receive $0$ points.



**Example Dataset Entry**

`Question:`
What nonlinearity is the default standard in fully connected neural networks?

`Answer:`
ReLU

Why this is a good question:
It refers to a [specific lecture](https://github.com/esokolov/ml-course-hse/blob/master/2025-fall/lecture-notes/lecture07-deeplearning.pdf).
Without access to the lecture notes, multiple answers could be possible.
However, when using the provided knowledge base, the correct answer is unambiguous.


In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

####🧾 **b) Prompt Design** ($1$ point)

Design a prompt for the RAG system.

The prompt must clearly specify the required answer format.



In [ ]:
# TODO: your code here

####🧹 **c) Post-processing and Normalization** ($1.5$ points)

Implement post-processing and normalization for the model’s answers.

For example:
 * remove trailing text
 * remove punctuation
 * normalize formatting

The goal is to make answers comparable to the reference answers.



In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

####📏 **d) Evaluation Metric** ($1$ point)

Use `cosine similarity` between embeddings of:
 * the reference answers
 * the normalized model outputs

Use **`FRIDA`** embeddings to compute these vectors.



In [ ]:
# TODO: your code here

###🧪 **Experiments** ($2$ points)

Conduct a series of experiments comparing the performance of the following systems:
 1. A **`RAG-based`** system
 2. An **`LLM without RAG`**, answering the same questions using prompting only
 3. A RAG system with the prompt written in **`another language`** (For example, if the original prompt is written in English, translate it into Russian (or vice versa)).

After running the experiments:
 * Analyze the results (no analysis will result in 0 points for the task)
 * Draw conclusions about the differences in performance
 * Propose ways to improve the system’s quality


In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**Your answers:** ...

### ⭐ **Bonus Task** (up to $3$ points)

Propose and implement a method to improve the quality of the system’s answers.

For example, you could:
 * convert the RAG system into an agent
 * allow it to use an API-based search tool
 * introduce other meaningful improvements to the pipeline

⚠️ `Simply changing one parameter, the prompt, or the model is not considered a substantial improvement.`

However, a well-justified combination of improvements (for example: selecting a better model, tuning generation parameters, and improving the prompt together) will count as a valid enhancement.

Your choice must be clearly explained and justified.

In [ ]:
# TODO: your code here

**Your answers here:** ...